In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("/content/DailyDelhiClimateTrain.csv")

In [3]:


df["lag_3"] = df["meantemp"].shift(3)
df["rolling_7"] = df["meantemp"].rolling(7).mean()

print(df.head())

         date   meantemp   humidity  wind_speed  meanpressure  lag_3  \
0  2013-01-01  10.000000  84.500000    0.000000   1015.666667    NaN   
1  2013-01-02   7.400000  92.000000    2.980000   1017.800000    NaN   
2  2013-01-03   7.166667  87.000000    4.633333   1018.666667    NaN   
3  2013-01-04   8.666667  71.333333    1.233333   1017.166667   10.0   
4  2013-01-05   6.000000  86.833333    3.700000   1016.500000    7.4   

   rolling_7  
0        NaN  
1        NaN  
2        NaN  
3        NaN  
4        NaN  


In [4]:


df = pd.DataFrame({
    "order_date": ["2025-01-01", "2025-01-04", "2025-01-05", "2025-01-06"]
})

df["order_date"] = pd.to_datetime(df["order_date"])

df["day_of_week"] = df["order_date"].dt.day_name()
df["month"] = df["order_date"].dt.month
df["weekend"] = df["order_date"].dt.dayofweek >= 5

print(df)

  order_date day_of_week  month  weekend
0 2025-01-01   Wednesday      1    False
1 2025-01-04    Saturday      1     True
2 2025-01-05      Sunday      1     True
3 2025-01-06      Monday      1    False


In [7]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


df = pd.read_csv("/content/DailyDelhiClimateTrain.csv")
df["lag_3"] = df["meantemp"].shift(3)
df["rolling_7"] = df["meantemp"].rolling(7).mean()

data = df.dropna()

X = data[["lag_3", "rolling_7"]]
y = data["meantemp"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

prediction = model.predict(X.tail(1))

print("Next Day Prediction:", prediction[0])

Next Day Prediction: 12.249345238095241


In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
mape = np.mean(np.abs((y_test - pred) / y_test)) * 100

print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape)

MAE: 1.392636119296317
RMSE: 1.7919264169764912
MAPE: 6.086292064137788


RMSE is the most sensitive to large errors because it squares the errors before averaging them.

In [9]:
import numpy as np

def smape(actual, predicted):
    return np.mean(
        2 * np.abs(actual - predicted) /
        (np.abs(actual) + np.abs(predicted))
    ) * 100

print("SMAPE:", smape(y_test.values, pred))

SMAPE: 6.042231470727918


I learned that SMAPE treats over-predictions and under-predictions equally and works better than MAPE when actual values are close to zero.